In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from mpl_toolkits.mplot3d import Axes3D
plt.style.use('seaborn-v0_8-whitegrid')

print("Libraries loaded successfully.")
print("This lab addresses: Climate impact on crop yields")
print("Core concept: Cost function for linear regression")

In [ ]:
# Temperature anomaly: how many degrees C above the historical baseline average
# during the growing season. Positive = warmer than normal.
# This is our INPUT variable (what we measure and observe).
x_train = np.array([1.0, 2.0])

# Yield change: percentage change in crop yield compared to historical baseline.
# Negative values mean yield DECREASED (bad for food security).
# Positive values mean yield INCREASED (possibly beneficial in cold regions).
# This is our TARGET variable (what we want to predict).
y_train = np.array([-6.0, -12.0])

print("CLIMATE IMPACT DATASET")
print("=" * 50)
print(f"{'Region':<10} {'Temp Anomaly':<15} {'Yield Change':<15}")
print("-" * 50)
for i in range(len(x_train)):
    print(f"{chr(65+i):<10} {x_train[i]:>+8.1f} C     {y_train[i]:>+8.1f} %")
print("=" * 50)
print(f"\nTotal observations: {len(x_train)}")
print(f"Units: Temperature in degrees Celsius, Yield in percentage")

In [ ]:
def compute_cost(x, y, w, b):
    """
    Computes the cost function for linear regression.
    
    This function measures how well parameters w and b predict the
    relationship between temperature anomalies and crop yield changes.
    
    Args:
      x (ndarray (m,)): Temperature anomaly values for m regions
      y (ndarray (m,)): Observed yield change percentages for m regions
      w (scalar): Model slope (yield sensitivity per degree C)
      b (scalar): Model intercept (baseline yield change at zero anomaly)
    
    Returns:
      total_cost (float): Average squared prediction error across all regions.
        Lower values indicate better predictions.
        A value of 0 means perfect prediction (rare with real data).
    
    Real-world interpretation:
        A cost of 0.5 means our average squared error is about 1.0 percent
        point in yield prediction. For a region producing 10 million tons of
        grain, that translates to 100,000 tons of potential miscalculation.
    """
    # Count the number of data points (regions observed)
    m = x.shape[0]
    
    # Accumulator for total squared error across all regions
    cost_sum = 0
    
    # Loop through each region's data point
    for i in range(m):
        # PREDICT: Apply linear model to get expected yield change
        # w controls sensitivity to temperature (steeper = more sensitive crops)
        # b controls baseline offset (could represent non-climate factors)
        f_wb = w * x[i] + b
        
        # ERROR: Difference between prediction and actual observation
        # If positive: model predicts MORE yield loss than reality (overly alarmist)
        # If negative: model predicts LESS yield loss than reality (dangerously optimistic)
        error = f_wb - y[i]
        
        # SQUARE: Eliminate sign, penalize large errors disproportionately
        # Squaring reflects the nonlinear human impact of prediction errors:
        # a 4% underestimate affects far more than twice as many people as a 2% error
        cost = error ** 2
        
        # ACCUMULATE: Add this region's error contribution to the total
        cost_sum = cost_sum + cost
    
    # NORMALIZE: Divide by 2m to get average cost per region
    # The factor of 2 is conventional and simplifies gradient calculations later
    total_cost = (1 / (2 * m)) * cost_sum
    
    return total_cost

In [ ]:
# DEMONSTRATION: How cost changes with different parameter values

print("COST FUNCTION DEMONSTRATION")
print("=" * 65)
print("Testing how different parameter values affect prediction quality.")
print("Recall: Region A (1.0 C, -6%) and Region B (2.0 C, -12%)")
print("=" * 65)

# Test several w values with b fixed at 0 (assume no offset for now)
b_fixed = 0
test_w_values = [-12, -6, -4, -3, 0, 3, 4, 6, 12]

print(f"\n{'w (slope)':<12} {'b (intercept)':<15} {'Cost J(w,b)':<15} {'Interpretation'}")
print("-" * 75)

for w_test in test_w_values:
    cost = compute_cost(x_train, y_train, w_test, b_fixed)
    
    if cost < 0.01:
        interpretation = "PERFECT FIT - predictions match exactly"
    elif cost < 1:
        interpretation = "Close approximation"
    elif cost < 10:
        interpretation = "Significant prediction error"
    elif cost < 50:
        interpretation = "Large errors - dangerous for planning"
    else:
        interpretation = "Extreme error - model is misleading"
    
    print(f"{w_test:>8.1f}     {b_fixed:>8.1f}       {cost:>8.2f}        {interpretation}")

print("\nKey Insight: The cost is minimized when w=-6, b=0")
print("This means each degree of warming reduces yield by 6%")
print("This matches our earlier observation from the data directly.")

In [ ]:
# VISUALIZATION 1: Data points with different model lines

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Define three scenarios to compare
scenarios = [
    {'w': -3.0, 'b': 0.0, 'title': 'Underestimate (w=-3)', 'color': 'orange'},
    {'w': -6.0, 'b': 0.0, 'title': 'Optimal (w=-6, cost=0)', 'color': 'green'},
    {'w': -12.0, 'b': 0.0, 'title': 'Overestimate (w=-12)', 'color': 'red'},
]

for ax_idx, scenario in enumerate(scenarios):
    ax = axes[ax_idx]
    
    # Plot actual data points
    ax.scatter(x_train, y_train, marker='o', c='blue', s=150, zorder=5, edgecolors='black', label='Observed Data')
    
    # Plot model prediction line
    x_line = np.linspace(0.5, 2.5, 100)
    y_line = scenario['w'] * x_line + scenario['b']
    ax.plot(x_line, y_line, c=scenario['color'], linewidth=2, label=f"Model: w={scenario['w']}")
    
    # Draw error bars (vertical dashed lines showing prediction vs actual)
    for i in range(len(x_train)):
        pred = scenario['w'] * x_train[i] + scenario['b']
        ax.plot([x_train[i], x_train[i]], [y_train[i], pred], 'k--', alpha=0.5, linewidth=1)
        
        # Add error annotation
        error = pred - y_train[i]
        if abs(error) > 0.01:
            mid_y = (y_train[i] + pred) / 2
            ax.annotate(f'{error:+.1f}%', xy=(x_train[i]+0.05, mid_y), fontsize=9, color='red')
    
    # Calculate and display cost
    cost = compute_cost(x_train, y_train, scenario['w'], scenario['b'])
    
    ax.set_title(f"{scenario['title']}\nCost J = {cost:.2f}", fontsize=12, fontweight='bold')
    ax.set_xlabel('Temperature Anomaly (degrees C)', fontsize=11)
    ax.set_ylabel('Yield Change (%)', fontsize=11)
    ax.legend(loc='lower left', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='gray', linewidth=0.5, linestyle=':')
    ax.set_xlim(0.5, 2.5)
    ax.set_ylim(-16, 2)

plt.suptitle('Impact of Parameter Choice on Prediction Quality', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Dashed black lines show prediction errors for each data point.")
print("Red numbers indicate the prediction error in percentage points.")
print("The center panel achieves zero cost: perfect predictions.")
print("Left and right panels show how wrong parameters mislead policymakers.")

In [ ]:
# VISUALIZATION 2: Cost curve as a function of w (with b=0)

# Generate a range of w values to plot the cost landscape
w_range = np.linspace(-15, 3, 300)
b_fixed = 0
costs = [compute_cost(x_train, y_train, w, b_fixed) for w in w_range]

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(w_range, costs, 'b-', linewidth=2, label='Cost J(w)')

# Mark the minimum
w_min = -6.0
cost_min = compute_cost(x_train, y_train, w_min, b_fixed)
ax.plot(w_min, cost_min, 'go', markersize=15, zorder=5, label=f'Minimum at w={w_min}')

# Mark a few specific points for discussion
highlight_points = [(-3, 'Underestimate'), (-12, 'Overestimate')]
for w_h, label in highlight_points:
    c_h = compute_cost(x_train, y_train, w_h, b_fixed)
    ax.plot(w_h, c_h, 'ro', markersize=10, zorder=5)
    ax.annotate(f'{label}\nCost={c_h:.1f}', xy=(w_h, c_h), xytext=(w_h-1, c_h+8),
                fontsize=9, arrowprops=dict(arrowstyle='->', color='red'),
                bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow'))

# Add zone labels
ax.axvspan(-15, -9, alpha=0.08, color='red', label='Danger: Overestimating impact')
ax.axvspan(-3, 0, alpha=0.08, color='orange', label='Danger: Underestimating impact')
ax.axvspan(-9, -3, alpha=0.08, color='green', label='Reasonable range')

ax.set_xlabel('w (Yield Sensitivity: % yield change per degree C)', fontsize=12)
ax.set_ylabel('Cost J(w, b=0)', fontsize=12)
ax.set_title('Cost Function Landscape: Finding Optimal Climate Sensitivity Parameter', fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='upper left')
ax.grid(True, alpha=0.3)
ax.set_ylim(-1, max(costs)*0.5)
plt.tight_layout()
plt.show()

print("The bowl-shaped curve is characteristic of the squared-error cost function.")
print("The minimum at w=-6 represents the best possible parameter for this data.")
print("Both overestimating (far left) and underestimating (right side) incur high cost.")
print("In policy terms: both excessive alarmism and complacency are costly.")

In [ ]:
# EXPANDED DATASET: Six regions with realistic scatter
# Data inspired by patterns in FAO and IPCC agricultural impact reports

x_train = np.array([1.0, 1.7, 2.0, 2.5, 3.0, 3.2])
y_train = np.array([-5.0, -8.0, -7.0, -16.0, -19.0, -21.0])

region_names = ['South Asia', 'Sub-Saharan Africa', 'Southern Europe', 
                'Central America', 'Southeast Asia', 'Horn of Africa']

print("EXPANDED CLIMATE IMPACT DATASET")
print("=" * 70)
print(f"{'Region':<25} {'Temp Anomaly':<15} {'Yield Change':<15}")
print("-" * 70)
for i in range(len(x_train)):
    print(f"{region_names[i]:<25} {x_train[i]:>+8.1f} C     {y_train[i]:>+8.1f} %")
print("-" * 70)
print(f"Total regions observed: {len(x_train)}")
print(f"\nNote: Data no longer falls on a perfect line.")
print(f"This reflects real-world variability in climate resilience.")

In [ ]:
# VISUALIZATION 3: Plot the expanded dataset

fig, ax = plt.subplots(figsize=(10, 7))

# Color-code points by severity
severity_colors = []
for y in y_train:
    if y > -8:
        severity_colors.append('green')
    elif y > -15:
        severity_colors.append('orange')
    else:
        severity_colors.append('red')

scatter = ax.scatter(x_train, y_train, c=severity_colors, s=200, 
                     edgecolors='black', zorder=5)

# Add region name annotations
for i, name in enumerate(region_names):
    offset_x = 0.08
    offset_y = 0.8
    ax.annotate(name, xy=(x_train[i], y_train[i]), 
                xytext=(x_train[i]+offset_x, y_train[i]+offset_y),
                fontsize=9, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

# Add severity legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='green', edgecolor='black', label='Low Impact (> -8%)'),
    Patch(facecolor='orange', edgecolor='black', label='Moderate Impact (-8% to -15%)'),
    Patch(facecolor='red', edgecolor='black', label='Severe Impact (< -15%)'),
]
ax.legend(handles=legend_elements, loc='lower left', fontsize=10)

ax.set_xlabel('Temperature Anomaly (degrees C above baseline)', fontsize=12)
ax.set_ylabel('Crop Yield Change (%)', fontsize=12)
ax.set_title('Climate Impact on Crop Yields: Six Global Regions', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='gray', linewidth=1, linestyle='--', alpha=0.5)
ax.set_xlim(0.5, 3.7)
ax.set_ylim(-25, 2)

# Add annotation explaining the chart
ax.text(0.55, 1.5, 'Values below zero = yield LOSS\nEach point = one region\nColor = severity category', 
        fontsize=9, style='italic', 
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

print("Green points: relatively resilient regions")
print("Orange points: moderate yield losses")
print("Red points: severe losses requiring urgent intervention")

In [ ]:
# COST COMPARISON: Test multiple parameter combinations on expanded data

print("PARAMETER EXPLORATION ON REALISTIC DATA")
print("=" * 70)
print("With scattered data, zero cost is impossible. We seek the minimum.")
print("=" * 70)

# Test a grid of w and b values
param_combinations = [
    (-5, 0),    # Simple guess
    (-7, 2),    # Moderate slope, positive intercept
    (-8, -1),   # Steeper slope
    (-6, -2),   # Less steep, negative intercept
    (-7.5, 0.5),# Refined guess
    (-6.5, -1.5),# Another attempt
]

print(f"\n{'w':<8} {'b':<8} {'Cost J(w,b)':<15} {'Avg Error':<12} {'Assessment'}")
print("-" * 65)

best_cost = float('inf')
best_params = None

for w_test, b_test in param_combinations:
    cost = compute_cost(x_train, y_train, w_test, b_test)
    avg_error = (2 * cost) ** 0.5  # Approximate RMS error
    
    if cost < best_cost:
        best_cost = cost
        best_params = (w_test, b_test)
        assessment = "<-- Best so far"
    elif cost < 2:
        assessment = "Good fit"
    elif cost < 5:
        assessment = "Acceptable"
    else:
        assessment = "Poor fit"
    
    print(f"{w_test:>6.1f}   {b_test:>6.1f}   {cost:>8.3f}         {avg_error:>6.2f}%      {assessment}")

print("\n" + "=" * 65)
print(f"Best parameters found: w={best_params[0]}, b={best_params[1]}")
print(f"Minimum cost achieved: {best_cost:.4f}")
print(f"Average prediction error: ~{(2*best_cost)**0.5:.2f}% yield")
print("=" * 65)

In [ ]:
# VISUALIZATION 4: 3D Cost Surface

# Create a grid of w and b values
w_values = np.linspace(-12, 0, 100)
b_values = np.linspace(-10, 10, 100)
W, B = np.meshgrid(w_values, b_values)

# Compute cost for every (w, b) combination
J = np.zeros_like(W)
for i in range(W.shape[0]):
    for j in range(W.shape[1]):
        J[i, j] = compute_cost(x_train, y_train, W[i, j], B[i, j])

# Find minimum cost location
min_idx = np.unravel_index(np.argmin(J), J.shape)
w_opt = W[min_idx]
b_opt = B[min_idx]
j_opt = J[min_idx]

fig = plt.figure(figsize=(16, 7))

# Left: 3D surface plot
ax1 = fig.add_subplot(121, projection='3d')
surf = ax1.plot_surface(W, B, J, cmap='viridis', alpha=0.8, edgecolor='none')
ax1.scatter(w_opt, b_opt, j_opt, color='red', s=100, marker='*', zorder=5,
            label=f'Minimum: w={w_opt:.1f}, b={b_opt:.1f}')
ax1.set_xlabel('w (slope)', fontsize=11)
ax1.set_ylabel('b (intercept)', fontsize=11)
ax1.set_zlabel('J(w,b) (cost)', fontsize=11)
ax1.set_title('3D Cost Surface\n(The Soup Bowl Shape)', fontsize=13, fontweight='bold')
ax1.view_init(elev=30, azim=45)
ax1.legend(fontsize=9)

# Right: 2D contour plot
ax2 = fig.add_subplot(122)
contour = ax2.contour(W, B, J, levels=30, cmap='viridis')
ax2.contourf(W, B, J, levels=30, cmap='viridis', alpha=0.6)
ax2.plot(w_opt, b_opt, 'r*', markersize=20, zorder=5, label=f'Minimum: w={w_opt:.1f}, b={b_opt:.1f}')

# Add a few parameter combinations from our test
for w_t, b_t in [(-5, 0), (-7, 2), (-8, -1)]:
    c = compute_cost(x_train, y_train, w_t, b_t)
    ax2.plot(w_t, b_t, 'wo', markersize=8, markeredgecolor='black')
    ax2.annotate(f'J={c:.1f}', xy=(w_t, b_t), xytext=(w_t+0.3, b_t+0.5), fontsize=8, color='white')

plt.colorbar(contour, ax=ax2, label='Cost J(w,b)')
ax2.set_xlabel('w (slope)', fontsize=11)
ax2.set_ylabel('b (intercept)', fontsize=11)
ax2.set_title('2D Contour Map of Cost Surface', fontsize=13, fontweight='bold')
ax2.legend(fontsize=9, loc='upper right')

plt.suptitle('Cost Function Visualization: Climate-Yield Model Optimization', 
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f"Optimal parameters: w = {w_opt:.2f}, b = {b_opt:.2f}")
print(f"Minimum cost: J = {j_opt:.4f}")
print(f"Since minimum cost > 0, the data has inherent scatter (irreducible error).")
print(f"This scatter represents factors beyond temperature: rainfall, soil, policy, etc.")

In [ ]:
# VISUALIZATION 5: Convex bowl demonstration with simplified symmetric scaling

fig = plt.figure(figsize=(14, 6))

# Create symmetric cost surface (scaled w and b equally)
w_sym = np.linspace(-10, 10, 150)
b_sym = np.linspace(-10, 10, 150)
W_sym, B_sym = np.meshgrid(w_sym, b_sym)

# Use a simple quadratic cost for illustration
# This represents the idealized symmetric bowl
J_sym = 0.5 * (W_sym**2 + B_sym**2) + 5

# 3D view
ax1 = fig.add_subplot(121, projection='3d')
surf = ax1.plot_surface(W_sym, B_sym, J_sym, cmap='coolwarm', alpha=0.7)
ax1.scatter(0, 0, 5, color='gold', s=150, marker='*', zorder=5, label='Global Minimum')
ax1.set_xlabel('w', fontsize=12)
ax1.set_ylabel('b', fontsize=12)
ax1.set_zlabel('J(w,b)', fontsize=12)
ax1.set_title('Symmetric Convex Cost Surface\n(The Ideal Soup Bowl)', fontsize=13, fontweight='bold')
ax1.view_init(elev=25, azim=35)
ax1.legend(fontsize=10)

# Contour view
ax2 = fig.add_subplot(122)
contour = ax2.contourf(W_sym, B_sym, J_sym, levels=30, cmap='coolwarm')
ax2.contour(W_sym, B_sym, J_sym, levels=30, colors='black', linewidths=0.5, alpha=0.3)
ax2.plot(0, 0, '*', color='gold', markersize=20, markeredgecolor='black', label='Global Minimum')

# Draw a hypothetical optimization path
path_w = [8, 6, 4, 2.5, 1.5, 0.8, 0.3, 0]
path_b = [-7, -5, -3, -2, -1, -0.5, -0.2, 0]
ax2.plot(path_w, path_b, 'g-o', linewidth=2, markersize=5, label='Gradient Descent Path')

plt.colorbar(contour, ax=ax2, label='Cost J(w,b)')
ax2.set_xlabel('w', fontsize=12)
ax2.set_ylabel('b', fontsize=12)
ax2.set_title('Contour View with\nOptimization Path', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10, loc='upper left')

plt.suptitle('Convexity Guarantees: Every Path Leads to the Same Minimum', 
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("Left: The 3D bowl shape. Notice it curves smoothly upward in all directions.")
print("Right: Contour rings with a sample gradient descent path (green dots).")
print("No matter where you start on this surface, descending always reaches the gold star.")
print("\nFor climate modeling, this means: our optimization is reliable and repeatable.")
print("Different researchers using different starting points will find the same model.")

In [ ]:
# FINAL APPLICATION: Using the optimized model for climate planning

# Use near-optimal parameters from our exploration
w_final = w_opt
b_final = b_opt

print("FINAL MODEL: CLIMATE-YIELD PREDICTION SYSTEM")
print("=" * 60)
print(f"Model equation: yield_change = {w_final:.2f} * temp_anomaly + {b_final:.2f}")
print(f"Cost (prediction error): J = {j_opt:.4f}")
print(f"Interpretation: Each degree C of warming causes approximately")
print(f"  {abs(w_final):.1f}% reduction in crop yield.")
print("=" * 60)

# Predict for several future climate scenarios
print("\nCLIMATE SCENARIO FORECASTS:")
print("-" * 60)

scenarios = [
    (0.5, "Low-warming scenario (Paris Agreement target)"),
    (1.5, "Moderate warming (current trajectory)"),
    (2.5, "High warming (business-as-usual path)"),
    (3.5, "Extreme warming (worst-case projections)"),
]

print(f"{'Scenario':<50} {'Temp':<8} {'Predicted Yield':<15}")
print("-" * 73)
for temp, desc in scenarios:
    predicted_yield = w_final * temp + b_final
    severity = "Manageable" if predicted_yield > -10 else "Serious" if predicted_yield > -20 else "Critical"
    print(f"{desc:<50} {temp:>4.1f}C  {predicted_yield:>+8.1f}%      [{severity}]")

print("-" * 73)
print("\nHuman Impact Translation:")
for temp, desc in scenarios:
    predicted_yield = w_final * temp + b_final
    # Assume a region producing 50 million tons annually
    tons_lost = abs(predicted_yield) / 100 * 50
    # Rough estimate: 1 ton feeds ~1700 people for a year
    people_affected = int(tons_lost * 1700)
    print(f"  {desc}: ~{tons_lost:.1f}M tons lost -> ~{people_affected/1e6:.1f}M people at risk")